# VAE latent-space comparison

Original notebook content is retained below. New latent-activity diagnostics are appended at the end.

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def find_repo_root(start):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from domain_knowledge_analysis import utils
DEVICE = utils.get_device()
print(f"Repository: {REPO_ROOT}")
print(f"Device: {DEVICE}")

MODEL_A_CHECKPOINT = (
    REPO_ROOT
    / "runs/vae_mnist_lr_0.001_24_jul_1720/checkpoints/last.pt"
)
MODEL_B_CHECKPOINT = (
    REPO_ROOT
    / "runs/vae_mnist_lr_0.001_24_jul_0928_B/checkpoints/last.pt"
)

# To compare a second model, use for example:
# MODEL_B_CHECKPOINT = REPO_ROOT / "runs/.../checkpoints/best.pt"

MODEL_A_NAME = "Model B"
MODEL_B_NAME = "Model CB"

DATA_SPLIT = "test"  # "train" or "test"
BATCH_SIZE = 256
NUM_PRIOR_SAMPLES = 12
NUM_CLASS_SAMPLES = 10
INTERPOLATION_STEPS = 12
INTERPOLATION_PAIRS = [(0, 1), (1, 7), (3, 8), (4, 9), (5, 6)]

# "empirical_mixture" samples the encoded class mixture directly.
# "moment_gaussian" uses one moment-matched diagonal Normal per class.
CLASS_SAMPLING_METHOD = "empirical_mixture"
RANDOM_SEED = 42

def load_vae_checkpoint(checkpoint_path, name):
    checkpoint_path = Path(checkpoint_path).expanduser().resolve()
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}\n"
            "Edit MODEL_A_CHECKPOINT and MODEL_B_CHECKPOINT in the settings cell."
        )

    # Only load checkpoints that you trust; torch checkpoints use pickle.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    config = checkpoint["config"]

    model = utils.create_model(config)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model = model.to(DEVICE).eval()

    return {
        "name": name,
        "path": checkpoint_path,
        "model": model,
        "config": config,
        "checkpoint": checkpoint,
        "latent_dim": config["model"]["encoder"]["latent_dim"],
        "distribution": config.get("loss", {}).get(
            "log_prob_function", "unspecified"
        ),
    }


model_specs = [
    (MODEL_A_CHECKPOINT, MODEL_A_NAME),
    (MODEL_B_CHECKPOINT, MODEL_B_NAME),
]
models = [
    load_vae_checkpoint(path, name)
    for path, name in model_specs
    if path is not None
]
if not models:
    raise ValueError("At least one checkpoint path must be provided.")

reference_dataset = models[0]["config"]["dataset"]
for item in models[1:]:
    assert item["config"]["dataset"]["name"] == reference_dataset["name"]
    assert item["config"]["dataset"]["shape"] == reference_dataset["shape"]

for item in models:
    checkpoint = item["checkpoint"]
    print(
        f"{item['name']}: "
        f"distribution={item['distribution']}, "
        f"latent_dim={item['latent_dim']}, "
        f"epoch={checkpoint.get('epoch', 'unknown')}, "
        f"validation_loss={checkpoint.get('validation_loss', 'unknown')}"
    )
    print(f"  {item['path']}")

evaluation_config = deepcopy(models[0]["config"])
evaluation_config.setdefault("loss", {})["log_prob_function"] = "bernoulli"

dataset_name = evaluation_config["dataset"]["name"]
evaluation_dataset = utils.create_dataset(
    config=evaluation_config,
    dataset_name=dataset_name,
    train=(DATA_SPLIT == "train"),
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(
    f"Using {len(evaluation_dataset)} raw {dataset_name.upper()} "
    f"{DATA_SPLIT} images for {len(models)} model(s)."
)

@torch.no_grad()
def encode_dataset(model, data_loader, description):
    model.eval()
    means = []
    log_variances = []
    labels = []

    for images, batch_labels in tqdm(data_loader, desc=description):
        mean, log_variance = model.encoder(images.to(DEVICE))
        means.append(mean.cpu())
        log_variances.append(log_variance.cpu())
        labels.append(batch_labels.cpu())

    mean = torch.cat(means, dim=0)
    log_variance = torch.cat(log_variances, dim=0)
    return {
        "mean": mean,
        "log_variance": log_variance,
        "std": torch.exp(0.5 * log_variance),
        "labels": torch.cat(labels, dim=0),
    }


def get_class_statistics(encoded):
    means = encoded["mean"]
    log_variances = encoded["log_variance"]
    labels = encoded["labels"]
    kl_per_image = 0.5 * (
        log_variances.exp()
        + means.square()
        - 1.0
        - log_variances
    ).sum(dim=1)

    statistics = {}
    for class_id in sorted(labels.unique().tolist()):
        indices = torch.where(labels == class_id)[0]
        class_means = means[indices]
        class_log_variances = log_variances[indices]

        aggregate_mean = class_means.mean(dim=0)
        aggregate_second_moment = (
            class_log_variances.exp() + class_means.square()
        ).mean(dim=0)
        aggregate_variance = (
            aggregate_second_moment - aggregate_mean.square()
        ).clamp_min(1e-8)

        statistics[int(class_id)] = {
            "indices": indices,
            "mean": aggregate_mean,
            "std": aggregate_variance.sqrt(),
            "mean_kl": kl_per_image[indices].mean(),
            "count": len(indices),
        }

    return statistics

for item in models:
    item["encoded"] = encode_dataset(
        item["model"],
        evaluation_loader,
        description=f"Encoding {item['name']}",
    )
    item["class_statistics"] = get_class_statistics(item["encoded"])

reference_labels = models[0]["encoded"]["labels"]
for item in models[1:]:
    assert torch.equal(reference_labels, item["encoded"]["labels"])

class_sets = [set(item["class_statistics"]) for item in models]
class_ids = sorted(set.intersection(*class_sets))
print(f"Classes: {class_ids}")

def print_class_summary(item):
    print(f"\n{item['name']} — {item['distribution']}")
    print("class | count | ||class mean|| | mean aggregate std | mean KL")
    print("------|-------|----------------|--------------------|--------")
    for class_id in class_ids:
        stats = item["class_statistics"][class_id]
        print(
            f"{class_id:>5} | "
            f"{stats['count']:>5} | "
            f"{stats['mean'].norm().item():>14.3f} | "
            f"{stats['std'].mean().item():>18.3f} | "
            f"{stats['mean_kl'].item():>7.3f}"
        )


for item in models:
    print_class_summary(item)

@torch.no_grad()
def decode_parameter_images(model, latents):
    """Decode the observation-distribution parameter image."""
    model.eval()
    logits = model.decoder(latents.to(DEVICE))
    images = torch.sigmoid(logits)
    return images.clamp(0.0, 1.0).cpu()


def show_image(axis, image, title=None):
    axis.imshow(image.squeeze().numpy(), cmap="gray", vmin=0.0, vmax=1.0)
    axis.axis("off")
    if title is not None:
        axis.set_title(title)


def sample_class_latents(item, class_id, n_samples, generator, method):
    encoded = item["encoded"]
    stats = item["class_statistics"][class_id]

    if method == "empirical_mixture":
        class_indices = stats["indices"]
        selected = class_indices[
            torch.randint(
                low=0,
                high=len(class_indices),
                size=(n_samples,),
                generator=generator,
            )
        ]
        noise = torch.randn(
            n_samples,
            item["latent_dim"],
            generator=generator,
        )
        return (
            encoded["mean"][selected]
            + encoded["std"][selected] * noise
        )

    if method == "moment_gaussian":
        noise = torch.randn(
            n_samples,
            item["latent_dim"],
            generator=generator,
        )
        return stats["mean"] + stats["std"] * noise

    raise ValueError(
        "CLASS_SAMPLING_METHOD must be 'empirical_mixture' "
        "or 'moment_gaussian'."
    )

figure, axes = plt.subplots(
    nrows=len(models),
    ncols=NUM_PRIOR_SAMPLES,
    figsize=(1.45 * NUM_PRIOR_SAMPLES, 1.8 * len(models) + 0.8),
    squeeze=False,
)

for row, item in enumerate(models):
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    latents = torch.randn(
        NUM_PRIOR_SAMPLES,
        item["latent_dim"],
        generator=generator,
    )
    images = decode_parameter_images(item["model"], latents)

    for column, image in enumerate(images):
        show_image(axes[row, column], image)
    axes[row, 0].set_ylabel(item["name"], rotation=0, labelpad=45, va="center")

figure.suptitle("Standard-normal prior decoder outputs")
plt.tight_layout(rect=(0.03, 0.0, 1.0, 0.93))
plt.show()
plt.close(figure)

figure, axes = plt.subplots(
    nrows=len(models),
    ncols=len(class_ids),
    figsize=(1.55 * len(class_ids), 1.8 * len(models) + 0.8),
    squeeze=False,
)

for row, item in enumerate(models):
    prototype_latents = torch.stack(
        [item["class_statistics"][class_id]["mean"] for class_id in class_ids]
    )
    prototype_images = decode_parameter_images(item["model"], prototype_latents)

    for column, (class_id, image) in enumerate(zip(class_ids, prototype_images)):
        show_image(axes[row, column], image, title=str(class_id) if row == 0 else None)
    axes[row, 0].set_ylabel(item["name"], rotation=0, labelpad=45, va="center")

figure.suptitle("Decoded class prototypes")
plt.tight_layout(rect=(0.03, 0.0, 1.0, 0.93))
plt.show()
plt.close(figure)

for class_id in class_ids:
    figure, axes = plt.subplots(
        nrows=len(models),
        ncols=NUM_CLASS_SAMPLES + 1,
        figsize=(1.4 * (NUM_CLASS_SAMPLES + 1), 1.8 * len(models) + 0.8),
        squeeze=False,
    )

    for row, item in enumerate(models):
        generator = torch.Generator().manual_seed(
            RANDOM_SEED + 1000 * row + class_id
        )
        sampled_latents = sample_class_latents(
            item,
            class_id=class_id,
            n_samples=NUM_CLASS_SAMPLES,
            generator=generator,
            method=CLASS_SAMPLING_METHOD,
        )
        prototype = item["class_statistics"][class_id]["mean"].unsqueeze(0)
        latents = torch.cat([prototype, sampled_latents], dim=0)
        images = decode_parameter_images(item["model"], latents)

        for column, image in enumerate(images):
            title = "prototype" if row == 0 and column == 0 else None
            show_image(axes[row, column], image, title=title)
        axes[row, 0].set_ylabel(
            item["name"],
            rotation=0,
            labelpad=45,
            va="center",
        )

    figure.suptitle(
        f"Class {class_id}: prototype and {CLASS_SAMPLING_METHOD} samples"
    )
    plt.tight_layout(rect=(0.03, 0.0, 1.0, 0.93))
    plt.show()
    plt.close(figure)

def linear_interpolation(start, end, steps):
    weights = torch.linspace(0.0, 1.0, steps).unsqueeze(1)
    return (1.0 - weights) * start.unsqueeze(0) + weights * end.unsqueeze(0)


for start_class, end_class in INTERPOLATION_PAIRS:
    if start_class not in class_ids or end_class not in class_ids:
        raise ValueError(
            f"Interpolation pair {(start_class, end_class)} is not in {class_ids}."
        )

    figure, axes = plt.subplots(
        nrows=len(models),
        ncols=INTERPOLATION_STEPS,
        figsize=(1.4 * INTERPOLATION_STEPS, 1.8 * len(models) + 0.8),
        squeeze=False,
    )

    for row, item in enumerate(models):
        start = item["class_statistics"][start_class]["mean"]
        end = item["class_statistics"][end_class]["mean"]
        path = linear_interpolation(start, end, INTERPOLATION_STEPS)
        images = decode_parameter_images(item["model"], path)

        for column, image in enumerate(images):
            if row == 0:
                if column == 0:
                    title = str(start_class)
                elif column == INTERPOLATION_STEPS - 1:
                    title = str(end_class)
                else:
                    title = f"{column / (INTERPOLATION_STEPS - 1):.1f}"
            else:
                title = None
            show_image(axes[row, column], image, title=title)

        axes[row, 0].set_ylabel(
            item["name"],
            rotation=0,
            labelpad=45,
            va="center",
        )

    figure.suptitle(f"Class {start_class} → class {end_class}")
    plt.tight_layout(rect=(0.03, 0.0, 1.0, 0.93))
    plt.show()
    plt.close(figure)


## Per-dimension latent activity and pruning diagnostics

For a diagonal Gaussian encoder
\[
q_\phi(z\mid x)=\mathcal N(\mu_\phi(x),\operatorname{diag}(\sigma_\phi^2(x))),
\]
a pruned or passive dimension should not be diagnosed from the variance of sampled \(z\) alone. If that dimension has collapsed to the standard-normal prior, sampled values still have variance near one.

The appended diagnostics therefore report, for every latent dimension \(j\):

- **Posterior-mean mean:** \(\mathbb E_x[\mu_j(x)]\).
- **Posterior-mean activity:** \(\operatorname{Var}_x[\mu_j(x)]\). The commonly used “active units” heuristic marks a dimension active when this exceeds \(0.01\).
- **Expected conditional variance:** \(\mathbb E_x[\sigma_j^2(x)]\).
- **Aggregate-posterior variance:**  
  \[
  \operatorname{Var}_{q(z)}(z_j)
  =
  \operatorname{Var}_x(\mu_j(x))
  +
  \mathbb E_x[\sigma_j^2(x)].
  \]
- **Mean per-dimension KL:**  
  \[
  \mathbb E_x\!\left[
  \tfrac12\left(\mu_j^2+\sigma_j^2-1-\log\sigma_j^2\right)
  \right].
  \]

A dimension is only labelled a **pruning candidate** when both posterior-mean activity and mean KL are very small. This is still a diagnostic, not proof that a smaller VAE will perform equally well; the final check is to retrain the smaller architecture and compare validation ELBO/reconstruction quality and prior samples.


In [ ]:
# Appended cell: per-dimension statistics for posterior collapse / pruning.
#
# The 0.01 cutoffs are conventional heuristics, not universal constants.
# Keep the continuous statistics and inspect their separation rather than
# relying only on the binary labels.

ACTIVE_MEAN_VARIANCE_THRESHOLD = 1e-2
ACTIVE_KL_THRESHOLD = 1e-2


def get_latent_dimension_statistics(encoded):
    """Compute analytic per-dimension statistics of q(z|x) and q(z)."""
    means = encoded["mean"].float()
    log_variances = encoded["log_variance"].float()
    posterior_variances = log_variances.exp()

    mean_of_posterior_means = means.mean(dim=0)
    variance_of_posterior_means = means.var(dim=0, unbiased=False)
    mean_posterior_variance = posterior_variances.mean(dim=0)

    # Law of total variance:
    # Var_q(z_j)[z_j] = Var_x(mu_j(x)) + E_x[sigma_j^2(x)].
    aggregate_posterior_variance = (
        variance_of_posterior_means + mean_posterior_variance
    )

    # Analytic KL(q(z_j|x) || N(0,1)), averaged over images.
    mean_kl_per_dimension = 0.5 * (
        posterior_variances
        + means.square()
        - 1.0
        - log_variances
    ).mean(dim=0)

    active_by_mean_variance = (
        variance_of_posterior_means > ACTIVE_MEAN_VARIANCE_THRESHOLD
    )
    active_by_kl = mean_kl_per_dimension > ACTIVE_KL_THRESHOLD

    # Conservative candidate: both common diagnostics say inactive.
    pruning_candidate = ~(active_by_mean_variance | active_by_kl)

    return {
        "mean_of_posterior_means": mean_of_posterior_means,
        "variance_of_posterior_means": variance_of_posterior_means,
        "mean_posterior_variance": mean_posterior_variance,
        "aggregate_posterior_variance": aggregate_posterior_variance,
        "mean_kl_per_dimension": mean_kl_per_dimension,
        "active_by_mean_variance": active_by_mean_variance,
        "active_by_kl": active_by_kl,
        "pruning_candidate": pruning_candidate,
    }


def print_latent_dimension_summary(item):
    stats = item["latent_dimension_statistics"]
    latent_dim = item["latent_dim"]

    print(f"\n{item['name']} — {item['distribution']}")
    print(
        "dim | E[mu]    | Var_x(mu) | E[sigma^2] | Var_q(z) | "
        "mean KL  | AU | KL-active | candidate"
    )
    print(
        "----|----------|-----------|------------|----------|"
        "----------|----|-----------|----------"
    )

    for dimension in range(latent_dim):
        print(
            f"{dimension:>3} | "
            f"{stats['mean_of_posterior_means'][dimension].item():>8.4f} | "
            f"{stats['variance_of_posterior_means'][dimension].item():>9.5f} | "
            f"{stats['mean_posterior_variance'][dimension].item():>10.5f} | "
            f"{stats['aggregate_posterior_variance'][dimension].item():>8.5f} | "
            f"{stats['mean_kl_per_dimension'][dimension].item():>8.5f} | "
            f"{str(bool(stats['active_by_mean_variance'][dimension])):>2} | "
            f"{str(bool(stats['active_by_kl'][dimension])):>9} | "
            f"{str(bool(stats['pruning_candidate'][dimension])):>9}"
        )

    active_au = torch.where(stats["active_by_mean_variance"])[0].tolist()
    active_kl = torch.where(stats["active_by_kl"])[0].tolist()
    candidates = torch.where(stats["pruning_candidate"])[0].tolist()

    print(
        f"\nActive units by Var_x(mu) > "
        f"{ACTIVE_MEAN_VARIANCE_THRESHOLD:g}: "
        f"{len(active_au)}/{latent_dim} -> {active_au}"
    )
    print(
        f"Active units by mean KL > {ACTIVE_KL_THRESHOLD:g}: "
        f"{len(active_kl)}/{latent_dim} -> {active_kl}"
    )
    print(
        f"Conservative pruning candidates: "
        f"{len(candidates)}/{latent_dim} -> {candidates}"
    )


for item in models:
    item["latent_dimension_statistics"] = (
        get_latent_dimension_statistics(item["encoded"])
    )
    print_latent_dimension_summary(item)


In [ ]:
# Appended cell: visualize each statistic instead of trusting only a cutoff.

for item in models:
    stats = item["latent_dimension_statistics"]
    dimensions = torch.arange(item["latent_dim"]).numpy()

    figure, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(14, 8),
        squeeze=False,
    )

    axes[0, 0].bar(
        dimensions,
        stats["mean_of_posterior_means"].numpy(),
    )
    axes[0, 0].axhline(0.0, linewidth=1)
    axes[0, 0].set_title(r"Posterior-mean location: $\mathbb{E}_x[\mu_j(x)]$")
    axes[0, 0].set_xlabel("latent dimension")
    axes[0, 0].set_ylabel("mean")

    axes[0, 1].bar(
        dimensions,
        stats["variance_of_posterior_means"].numpy(),
    )
    axes[0, 1].axhline(
        ACTIVE_MEAN_VARIANCE_THRESHOLD,
        linestyle="--",
        label=f"active-unit threshold = {ACTIVE_MEAN_VARIANCE_THRESHOLD:g}",
    )
    axes[0, 1].set_title(
        r"Posterior-mean activity: $\mathrm{Var}_x[\mu_j(x)]$"
    )
    axes[0, 1].set_xlabel("latent dimension")
    axes[0, 1].set_ylabel("variance")
    axes[0, 1].legend()

    axes[1, 0].bar(
        dimensions,
        stats["mean_kl_per_dimension"].numpy(),
    )
    axes[1, 0].axhline(
        ACTIVE_KL_THRESHOLD,
        linestyle="--",
        label=f"KL threshold = {ACTIVE_KL_THRESHOLD:g}",
    )
    axes[1, 0].set_title("Mean KL contribution per latent dimension")
    axes[1, 0].set_xlabel("latent dimension")
    axes[1, 0].set_ylabel("nats")
    axes[1, 0].legend()

    axes[1, 1].bar(
        dimensions,
        stats["mean_posterior_variance"].numpy(),
        label=r"$\mathbb{E}_x[\sigma_j^2(x)]$",
    )
    axes[1, 1].bar(
        dimensions,
        stats["variance_of_posterior_means"].numpy(),
        bottom=stats["mean_posterior_variance"].numpy(),
        label=r"$\mathrm{Var}_x[\mu_j(x)]$",
    )
    axes[1, 1].axhline(
        1.0,
        linestyle="--",
        label="standard-normal prior variance",
    )
    axes[1, 1].set_title(
        r"Aggregate variance decomposition: "
        r"$\mathrm{Var}_{q(z)}(z_j)$"
    )
    axes[1, 1].set_xlabel("latent dimension")
    axes[1, 1].set_ylabel("variance")
    axes[1, 1].legend()

    figure.suptitle(
        f"{item['name']} — per-dimension latent activity diagnostics"
    )
    plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
    plt.show()
    plt.close(figure)


### Optional decoder-use check

Low encoder activity and low KL are strong evidence of selective posterior collapse, but “can be removed” is a statement about the whole trained model. The optional cell below perturbs one latent coordinate at a time by \(\pm1\) around encoded posterior means and measures the average change in decoded parameter images. A candidate with near-zero decoder sensitivity is especially likely to be ignored.

This check is deliberately optional because it requires roughly \(2 \times\) latent-dimension decoder passes over the selected subset.


In [ ]:
RUN_DECODER_SENSITIVITY_CHECK = False
DECODER_SENSITIVITY_SAMPLES = 1024
DECODER_SENSITIVITY_BATCH_SIZE = 256
DECODER_PERTURBATION = 1.0


@torch.no_grad()
def get_decoder_dimension_sensitivity(
    item,
    max_samples=DECODER_SENSITIVITY_SAMPLES,
    batch_size=DECODER_SENSITIVITY_BATCH_SIZE,
    perturbation=DECODER_PERTURBATION,
):
    """
    Mean absolute pixel-parameter change caused by moving z_j by +/- perturbation.

    This checks whether the decoder locally responds to each coordinate. It is
    supporting evidence only; retraining a lower-dimensional model remains the
    definitive model-selection experiment.
    """
    model = item["model"]
    means = item["encoded"]["mean"][:max_samples]
    latent_dim = item["latent_dim"]

    total_change = torch.zeros(latent_dim)
    total_images = 0

    for start in tqdm(
        range(0, len(means), batch_size),
        desc=f"Decoder sensitivity: {item['name']}",
    ):
        latents = means[start:start + batch_size]
        current_batch_size = len(latents)

        for dimension in range(latent_dim):
            plus = latents.clone()
            minus = latents.clone()
            plus[:, dimension] += perturbation
            minus[:, dimension] -= perturbation

            decoded_plus = decode_parameter_images(model, plus)
            decoded_minus = decode_parameter_images(model, minus)

            per_image_change = (
                decoded_plus - decoded_minus
            ).abs().flatten(start_dim=1).mean(dim=1)

            total_change[dimension] += per_image_change.sum()

        total_images += current_batch_size

    return total_change / total_images


if RUN_DECODER_SENSITIVITY_CHECK:
    for item in models:
        sensitivity = get_decoder_dimension_sensitivity(item)
        item["decoder_dimension_sensitivity"] = sensitivity

        candidates = torch.where(
            item["latent_dimension_statistics"]["pruning_candidate"]
        )[0].tolist()

        print(f"\n{item['name']} decoder sensitivity")
        for dimension, value in enumerate(sensitivity.tolist()):
            marker = "candidate" if dimension in candidates else ""
            print(f"dim {dimension:>2}: {value:.7f} {marker}")

        figure, axis = plt.subplots(figsize=(12, 4))
        axis.bar(range(item["latent_dim"]), sensitivity.numpy())
        axis.set_title(
            f"{item['name']} — decoder sensitivity to ±"
            f"{DECODER_PERTURBATION:g} latent perturbations"
        )
        axis.set_xlabel("latent dimension")
        axis.set_ylabel("mean absolute decoded-pixel change")
        plt.tight_layout()
        plt.show()
        plt.close(figure)
